# RSA

In [ ]:
import random


RSA (Rivest–Shamir–Adleman, 1977) is a widely used asymmetric algorithm for secure communication.

The algorithm requires a pair of keys:

- **Public key** $k_{\text{pub}}$: used for encryption.
- **Private key** $k_{\text{priv}}$: used for decryption.

RSA is based on integer factorization:

- Plaintext $x$ and ciphertext $y$ are modeled as integers, where   $x, y \in \mathbb{Z}_n$.
- Security relies on the difficulty of factoring the product of two large prime numbers.

![Caption](images/rsa_enc_dec.png)

**RSA Key Generation**: The key is generated through five steps:

1. Choose two prime numbers, $p$ and $q$.
2. Compute $n = p \cdot q$
3. Compute $m = \phi(n) = (p - 1)(q - 1)$
4. Draw $e \in \mathbb{Z}_n$ such that $\gcd(e, m) = 1$.
5. Compute $d$ such that  $ d \cdot e \equiv 1 \pmod{m} $

Steps **4** and **5** require the **Extended Euclidean Algorithm (EEA)**.

**Practical Issues**

1. **Exponentiation Involving Large Numbers**:
    When dealing with very large numbers $a$ and $b$, for example  $a, b \in \mathbb{Z}_{2^{2048}}$, it is not trivial to compute $a^b $ because it may require an unworkable amount of time.
    To solve this problem, many algorithms for fast and efficient exponentiation have been studied. 
    **Square-and-Multiply** is the basis for most of them.

2. **Generation of Large Prime Numbers**:
    Testing for primality is a much easier task than integer factorization. Therefore, one can randomly draw a large number and then test it for primality.
    A well-known algorithm for testing primality is the **Miller-Rabin Test**.

**Requirement - Python Integers**

An RSA implementation should rely on Python integers, namely `int`.

Python allows integers to be arbitrarily large. The only practical limit is the amount of available memory.

This means that Python `int` supports:

- integers of thousands of bits;
- no overflow.

Conversely, NumPy integers have fixed limits defined by their `dtype`.

Therefore, NumPy integers are limited in size and may suffer from overflow.

# Extended Euclidean Algorithm 

EEA is used to compute the private exponent $d$ from the public exponent $e$ and $\phi(n)$.

Given two integers $a$ and $m$, such that $m > a$ and the $gcd(a,m)=1$, then it finds integers $s,t \in \mathbb{Z}$ such that
$$
\gcd(a,m) = s \cdot a + t \cdot m
$$

In case $\gcd(a,m) = 1$,
$$
1 = \gcd(a,m) = s \cdot a + t \cdot m \equiv s \cdot a \pmod{m}
$$
As a consequence, when $\gcd(a,m) = 1$, $s$ is the inverse of $e$ modulo $m$, i.e.,  $s \equiv e^{-1} \pmod{m}$.

Therefore, the Extended Euclidean Algorithm can be used to find $d$ as an inverse of $e$ modulo $m=\phi(n)$.


### Algorithm
$$
\begin{align*}
& \text{Inputs:}\; a, m \quad (\text{with}\; m > a) \\
& r0 \leftarrow m, r1 \leftarrow a \\
& s0 \leftarrow 0, s1 \leftarrow 1 \\
& t0 \leftarrow 1, t1 \leftarrow 0 \\
& \text{do} \\
& \qquad i \leftarrow i+1 \\
& \qquad r_i \leftarrow r_{i-2} \bmod r_{i-1} \\
& \qquad q_i \leftarrow (r_{i-2} - r_i) / r_{i-1} \\
& \qquad s_i \leftarrow s_{i-2} - q_i \cdot s_{i-1} \\
& \qquad t_i \leftarrow t_{i-2} - q_i \cdot t_{i-1} \\
& \text{while}\; r_i \ne 0 \\
& \text{Outputs:}\; r_{i-1}, s_{i-1}, t_{i-1}\\
\end{align*}
$$

In [ ]:
def extended_euclidean_algorithm_naive(a: int, m: int) -> tuple[int, int, int]:
    r0, r1 = m, a
    s0, t0, s1, t1 = 0, 1, 1, 0
    while r1 != 0:
        r = r0 % r1
        q = (r0 - r) // r1
        r0, r1 = r1, r
        s0, s1 = s1, s0 - q * s1
        t0, t1 = t1, t0 - q * t1
    return r0, s0, t0

def extended_euclidean_algorithm(a: int, m: int) -> tuple[int, int, int]:
    if a == 0:
        return m , 0, 1
    gcd, x, y = extended_euclidean_algorithm(m % a , a)
    return gcd, y -(m // a)* x, x


In [ ]:
a = 121
m = 231

a = 123456789123456789
m = 987654321987654321

gcd, s, t = extended_euclidean_algorithm(a, m)
print(f'gcd: {gcd}, s: {s} , t: {t}')

| bits | a | m | gcd | s | t |
|---|---:|---:|---:|---:|---:|
| 8 | 121 | 231 | 11 | 2 | -1 |
| 64 | 123456789123456789 | 987654321987654321 | 9000000009 | -8 | 1 |


**Using `pow`**: Python can compute the modular inverse directly with:

```python
pow(base, exponent, modulus)
```

Internally, this uses the extended Euclidean algorithm which, when it exists, finds an integer $s$ such that $s \cdot a \equiv 1 \pmod{m}$ and returns $ s \bmod m$

In [ ]:
a = 5
m = 192
inverse_pow = pow(a, -1, m)

gcd, s, t = extended_euclidean_algorithm(a, m)

if gcd != 1:
    raise ValueError("a has no inverse modulo m")
inverse_eea = s % m

print(inverse_pow, inverse_eea)

# Square-and-Multiply

Computes the modular exponentiation $x^e \bmod n$ using repeated squaring and multiplication.

- `Input`:
    - base $x$
    - exponent $e$
    - modulo $n$

- `Output`:
    - $y = x^e \bmod n$

In [ ]:
def square_and_multiply(base: int, exp: int, modulo: int)-> int:
    ''' Square-and-Multiply Algorithm for exponentiation '''
    result = base
    length = exp.bit_length()
    for i in reversed(range(length - 1)):
        result = result**2 % modulo
        if bool((exp >> i) & 1):
            result = (result * base) % modulo
    return result

In [ ]:
base, exp, modulo = 230, 57, 386
base, exp, modulo = 11813480175700702747, 1638165922617329871, 36361664189978586673

x = square_and_multiply(base, exp, modulo)
x

In [ ]:
# pow implements square and multiply too,
# you can use for exponentiation involving big numbers
x = pow(base, exp, modulo)
x

| bits | base | exp | mod | x |
|---:|---:|---:|---:|---:|
| 8 | 230 | 57 | 386 | 88 |
| 64 | 11813480175700702747 | 1638165922617329871 | 36361664189978586673 | 7951981921739512458 |
|

# RSA Encrypt - Decrypt

In [ ]:
from math import ceil

def _crypt(text: bytes, key: tuple[int, int]):
    modulo, exp = key
    base = int.from_bytes(text, byteorder='big')
    result = pow(base, modulo, exp)
    length = ceil(result.bit_length() / 8)
    return result.to_bytes(length, byteorder='big')


def rsa_encrypt(plaintext: bytes, public_key: tuple[int, int]) -> bytes:
    """ Encrypt the message using the public key. """
    return _crypt(plaintext, public_key)

def rsa_decrypt(ciphertext: bytes, private_key: tuple[int, int]) -> bytes:
    """ Decrypt the ciphertext using the private key. """
    return _crypt(ciphertext, private_key)

# Miller-Rabin Primality Test

Determines whether a given number is likely to be prime or surely composite using the Miller-Rabin primality test.

The general idea is to write `p - 1` as:

$$
p - 1 = q \cdot 2^r
$$

where `q` is odd. Then, for several random values `x`, the algorithm checks whether `x` satisfies the modular arithmetic property that prime numbers must satisfy.

For a prime number `p`, the value

$$
y = x^q \bmod p
$$

must either be:

$$
y = 1
$$

or, after repeatedly squaring `y` modulo `p`, it must eventually become:

$$
y = p - 1
$$

Each tested value `x` gives one round of evidence. If the condition fails in any round, then `p` is surely composite. passing more rounds increases confidence that `p` is prime, but does not prove it. 



- `Input`: 
    -   Candidate odd prime number 𝑝 (int)
    -   Number of trials 𝑁 (int)
- `Outputs`: 
    -   Whether 𝑝 is probably prime (True) or 𝑝 is surely composite (False)

In [ ]:
def odd_decomposition(p: int) -> tuple[int, int]:
    ''' Finds q and r such that p = q*2^r + 1 '''
    if not bool(p & 1):
        raise ValueError("p must be odd")
    r = 0
    q = p - 1
    for i in range(q.bit_length()):
        if bool(q & 1):
            break
        q = q >> 1
        r += 1
    return q, r

In [ ]:
p = 41
p = 13288407253494681411
q, r = odd_decomposition(p)
print(f"q = {q}, r = {r}")

| bits | p | q | r |
|---:|---:|---:|---:|
| 8 | 41 | 5 | 3 |
| 64 | 13288407253494681411 | 6644203626747340705 | 1 |

In [ ]:
def miller_rabin_test(p: int, N: int = 100) -> bool:

    if p in (2, 3):
        return True
    if p < 2 or p % 2 == 0:
        return False

    q , r = odd_decomposition(p)
    for _ in range(N):
        x = random.randint(2, p-1)
        y = pow(x, q, p)
        if y == 1 or y == p - 1:
            continue
        for _ in range(r-1):
            y = pow(y, 2, p)
            if y == p -1:
                break
        else:
            return False
    return True



In [ ]:
p = 18446744073709551555
is_prime = miller_rabin_test(p, N=100)
print("is prime?", is_prime)

| bits | p | Prime? |
|---:|---:|---:|
| 8 | 251 | True |
| 8 | 28 | False |
| 64 | 18446744073709551557 | True |
| 64 | 18446744073709551555 | False |

# Generate Prime Numbers

Implement a function that generates a random prime number with a specific number of bits.

In [ ]:
def generate_prime(size: int) -> int:
    """ Generate a prime number with the specified number of bits. """
    #CODE HERE
    return n

In [ ]:
size = 128
p = generate_prime(size)
p

# RSA class

Implement:

- RSADecryptor: the user must specify the key length, `length`, so that the public key `k_pub` and the private key `k_priv` can be generated.

- RSAEncryptor: the user must provide the public key, so that a message can be encrypted:

$$
k_{pub} = (n, e)
$$



In [ ]:
class RSAEncryptor:

    def __init__(self, public_key: tuple[int, int]):
        #CODE HERE
        self.e = ...
        self.n = ...

    def encrypt(self, plaintext: bytes) -> bytes:
        #CODE HERE
        return ciphertext


In [ ]:
class RSADecryptor:

    def __init__(self, length: int):
        #CODE HERE
        self.private_key = n, d
        self.public_key = n, e

    def decrypt(self, ciphertext: bytes) -> bytes:
        #CODE HERE
        return plaintext

class RSADecryptor:

    def __init__(self, length: int):
        # CODE HERE
        self.n = ...
        self.e = ...
        self.d = ...

    @property
    def public_key(self):
        return self.n, self.e

    @staticmethod
    def _generate_prime_numbers(size: int) -> tuple[int, int]:
        # CODE HERE
        return p, q

    @staticmethod
    def _generate_keys(modulo: int) -> tuple[int, int]:
        # CODE HERE
        return e, d

    def decrypt(self, ciphertext: bytes) -> bytes:
        # CODE HERE
        return plaintext

In [ ]:
random.seed(0)
length = 256
bob = RSADecryptor(length)
alice = RSAEncryptor(bob.public_key)

plaintext = b'Hello world!'

ciphertext = alice.encrypt(plaintext)
decrypted = bob.decrypt(ciphertext)

print(" plaintext:", plaintext)
print("ciphertext:", ciphertext)
print(" decrypted:", decrypted)



```text
 plaintext: b'Hello world!'
ciphertext: b'\x01Y-\xb5\x8d\x90\x88\x05\xfd\xda,\x04\xfb\xa5\x15\xdb1a\xe6\x05\xf1\x08\xc5j\xda\xa6\x85\xed\xbd\xa9)k|'
 decrypted: b'Hello world!'
```

# RSA + AES

RSA is not suited to provide confidentiality in case of large messages. However, it can be exploited to establish a secure channel over which two entities (Alice and 
Bob) can exchange the key for a symmetric algorithm.

![Caption](images/rsa_aes.png)

Implement two classes: `Sender` and `Receiver`.

Use a **hybrid encryption scheme**:

- **AES** encrypts the message.
- **RSA** encrypts the AES key.


The `Receiver` should:

1. Generate an RSA key pair.
2. Share its public key.
3. Decrypt the AES key using its RSA private key.
4. Decrypt the message using AES.

The `Sender` should:

1. Receive the receiver’s RSA public key.
2. Generate a random AES key.
3. Encrypt the message with AES.
4. Encrypt the AES key with RSA.
5. Return both encrypted values.



In [ ]:
from aes import AES

class Receiver:

    def __init__(self, length: int, iv: bytes, mode: str) -> None:
        #CODE HERE
        self.public_key = ...

    def receive_encrypted_key(self, encrypted_key: bytes) -> None:
        #CODE HERE

    def encrypt(self, plaintext: bytes) -> bytes:
        #CODE HERE
        return ciphertext

    def decrypt(self, ciphertext: bytes) -> bytes:
        #CODE HERE
        return plaintext


class Sender:

    def __init__(self, public_key: tuple[int, int], iv: bytes, mode: str) -> None:
        #CODE HERE
        self.encrypted_key = ...

    def encrypt(self, plaintext: bytes) -> bytes:
        #CODE HERE
        return ciphertext

    def decrypt(self, ciphertext: bytes) -> bytes:
        #CODE HERE
        return plaintext

In [ ]:
length = 128
mode = 'CBC'
iv = bytes(AES.block_size)

bob = Receiver(length, iv, mode)

alice = Sender(bob.public_key, iv, mode)

bob.receive_encrypted_key(alice.encrypted_key)

plaintext = b'Hello world!'
ciphertext = alice.encrypt(plaintext)
decrypted = bob.decrypt(ciphertext)

print(" plaintext:", plaintext)
print("ciphertext:", ciphertext)
print(" decrypted:", decrypted)

```text
 plaintext: b'Hello world!'
ciphertext: b'kJ\xd5\xa7q\x03\xef\xac\x96\x1aY"\x17\xeb\xa4\x06'
 decrypted: b'Hello world!'
```